## This is a tutorial for using the code in the file 'kuramoto_simulation.py'

First, make sure the conda environment from metronomes.yaml is created and activated. To do this run the following commands:

In [1]:
#! conda env create -f ../environments/metronomes.yaml
#! conda activate metronomes

Then create a new python or jupyter notebook, making sure that kuramoto_simulation.py is in the same directory.
To start making the simulation import the kuramoto simulation as such:

In [1]:
from kuramoto_simulation import Window, Screen_params, Model_params, Data_Collector, Solver, Standard_Step
import numpy as np

pygame 2.6.1 (SDL 2.32.56, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


### Explaining the different classes and their uses

#### Window: the timestep for the simulation

This class iterates the pygame window and handles the update at each timestep. <br>
To initialise a Window two parameters need to be defined: <br>
Screen_params and Model_params.

#### Screen_params:

The parameters for the drawing of the simulation, nothing to do with the maths, but only the visualisation.
It is a data class with the screen width, height, background colour and model radius defined:

In [2]:
# Defining Screen_params

# the width, height and radius must be ints. Background colour is an RGB value so must be a tuple of 3 ints.
screen_params = Screen_params(800, # width
                              800, # height
                              350, # radius
                              (0, 20, 80)) # background colour, RGB

#### Model_params:

The parameters for the model itself, defining the oscillators initial phases and natural frequencies as well as the update rule and coupling strength. <br>
The initial phases and natural frequencies are passed into Model_params as lists, so they can be generated by defining a function that takes in N, the number of oscillators, and returns the list of natural frequencies and initial phases.

In [3]:
# Defining the Model_params

model_params = Model_params(0.4, # K
                            [0.3, 0.5, 0.6], # the list of natural frequencies
                            [0.2, 0.46, 1.3], # the list of initial phases
                            Standard_Step) # the step function

# alternative with functions to build phases and frequencies

# example function that generates the natural frequencies
def generate_natural_frequencies(num_frequencies: int) -> list[float]:
   return np.random.uniform(0, 0.5, num_frequencies).tolist()

# example function that generates the initial angles as evenly spaced around the circle
def generate_initial_angles(num_angles: int) -> list[float]:
   return np.linspace(0, 2*np.pi, num=num_angles).tolist()


N = 15
model_params = Model_params(0.4, # K
                            generate_natural_frequencies(N), # the list of natural frequencies
                            generate_initial_angles(N), # the list of initial phases
                            Standard_Step) # the step function

#### The step function: how to implement control and coupling:

This is probably the most important thing to choose; the step function determines how the system evolves and how control is given to it.

I will first define the simple Kuramoto control and then show where control could be implemented.

In [4]:
# Building the new step - simple Kuramoto model with the possibility of control


# a generic control function that takes in the time and oscillator position and returns a control output
def control(t: float, Y: list[float] | np.ndarray) -> float:
   return 0

def new_step(t: float, Y: list[float] | np.ndarray, K: float, N: int, nat_freqs: list[float]) -> np.ndarray:
   
   # the derivatives - initialised as empty
   dYdt = []

   # then each oscillator is looped through and the updated angular velocity is added to dYdt
   for i in range(N):
      # this is where the control can be added - just added to the system response.
      d_theta_dt = nat_freqs[i] + K/N * sum(np.sin(Y[j] - Y[i]) for j in range(N)) + control(t, Y)
      dYdt.append(d_theta_dt)
   
   # the derivatives are then returned
   return np.array(dYdt)

This modified step function can then be included in the definition of the Model_params:


In [5]:
model_params = Model_params(0.4,
                            generate_natural_frequencies(N),
                            generate_initial_angles(N),
                            new_step)

### Window: where the model is built and run

The Window class takes as arguments the screen and model parameters (and an optional data collector) and then runs the simulation.

## DO NOT CALL THE MAIN FUNCTION IN THIS JUPYTER NOTEBOOK

## FOR SOME REASON IT KILLS THE KERNEL

## SO MAKE A SCRIPT THAT IMPORTS THE LIBRARY

In [ ]:
# To build the window just initialise everything as:

window = Window(screen_params,
                model_params)

# to run the model just run the command:

#! This model doesn't run very nicely in a notebook - it is more stable when run in a python script !#
window.main()


Oscillators:
Natural Frequency: 0.22206844518840113, Angle: 0.3543863819215568
Natural Frequency: 0.05371157227278722, Angle: 0.5114531000649825
Natural Frequency: 0.3267994387582902, Angle: 1.3875193461609743
Natural Frequency: 0.34013994051040525, Angle: 1.854742989208415
Natural Frequency: 0.3218887037866686, Angle: 2.2799547142143735
Natural Frequency: 0.43053453727313984, Angle: 2.9281258132528047
Natural Frequency: 0.08018905010281507, Angle: 2.806408108979067
Natural Frequency: 0.15500003428189957, Angle: 3.4057090003915937
Natural Frequency: 0.14588201307652426, Angle: 3.8611980202299225
Natural Frequency: 0.4018233548777101, Angle: 4.750789869499535
Natural Frequency: 0.49976386737096534, Angle: 5.360390818113027
Natural Frequency: 0.16409370311681037, Angle: 5.254914160759388
Natural Frequency: 0.18680112281947797, Angle: 5.72651463147189
Natural Frequency: 0.26810591221778596, Angle: 6.285268063014581
Natural Frequency: 0.010279175567943777, Angle: 6.298132815945538

time e

: 

### Data_collector: how to get and save data from the model

This is a class that is defined by YOU; it is initialised when the model is run, updated at every step and then finally when exited.
Then the data can be taken from the model and turned into graphs, csv files, extra data (coupling parameters, average phases etc...) <br>
To make a datacollector first define the Data_Collector class <br>

The datacollector has three methods:
- ```start()```: this is called when all the oscillators are created and the solver is initialised. It takes two arguments: a window object and a solver object. This allows a snapshot of the model at the start to be made and is a good way to initialise all the data collector variables.
- ```collect()```: this is called at each new frame when the model is updated. It takes two arguments: window and solver. This method is used to update the data in the data collector.
- ```get_data()```: this method is called when you want to actually get data from the model after the model has finished running.

For this example a simple data collector just finds the position of each oscillator at each time step.

In [ ]:
class collector(Data_Collector):
   def __init__(self):
      super().__init__()
   def start(self, solver: Solver, window: Window) -> None:
      self.oscillators = window.oscillators
   def collect(self, solver: Solver, window: Window) -> None:
      self.oscillators.append(window.oscillators)
   def get_data(self) -> list[int]:
      return self.oscillators

And that's it! pass an instance of the collector into the setup for the window object and then you can collect data to draw graphs and calculate metrics. <br>